# 05 · Prompt Evaluation
### Practical LLM Engineering — Module 02: Prompt Engineering

> **What you'll learn**
> - Why systematic evaluation is the foundation of reliable prompt engineering
> - Building a labelled eval dataset and a reproducible test harness
> - Reference-based metrics: exact match, F1, BLEU, ROUGE
> - Model-based evaluation: LLM-as-a-judge
> - A/B testing prompts with statistical significance
> - Regression testing and prompt versioning
> - Engineering insights: eval cost, coverage, and the eval pyramid

---


## 1. Overview

**Prompt engineering without evaluation is just guessing.**

The discipline that separates production prompt engineering from ad-hoc experimentation is rigorous measurement:

```
Without evaluation:           With evaluation:
  "This prompt feels better"    "Prompt v2 improved accuracy by +8.3pp
                                  on the held-out test set (p < 0.05)"
```

Evaluation serves three purposes:

1. **Development** — fast feedback loop when iterating on prompts
2. **Regression testing** — catch quality drops when models or prompts change
3. **A/B testing** — choose between competing prompt strategies with confidence

### The evaluation stack

```
┌─────────────────────────────────────────────────┐
│  LLM-as-a-Judge (slow, expensive, flexible)     │ ← open-ended tasks
├─────────────────────────────────────────────────┤
│  Reference metrics: BLEU, ROUGE, BERTScore      │ ← generation tasks
├─────────────────────────────────────────────────┤
│  Exact match / F1 / Accuracy                    │ ← classification, extraction
└─────────────────────────────────────────────────┘
          ↑ fast, cheap, deterministic
```

### Module position

```
01_zero_shot_prompting   ✅
02_few_shot_prompting    ✅
03_chain_of_thought      ✅
04_structured_prompting  ✅
05_prompt_evaluation     ← you are here  (Module 02 finale)
```


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install anthropic openai rouge-score nltk numpy matplotlib scipy -q
# import nltk; nltk.download('punkt')

import os, re, json, time, math, random, hashlib, textwrap
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, Callable, Any
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

print("✅ Libraries ready")


In [ ]:
# ── LLM client abstraction (from notebook 01) ────────────────────────
@dataclass
class LLMResponse:
    text: str; model: str; input_tokens: int
    output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class BaseLLMClient(ABC):
    @abstractmethod
    def complete(self, system, user, max_tokens=512, temperature=0.0) -> LLMResponse: ...
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

class ClaudeClient(BaseLLMClient):
    def __init__(self, model="claude-sonnet-4-20250514", api_key=None):
        import anthropic
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.messages.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{"role":"user","content":user}])
        return LLMResponse(msg.content[0].text, self.model,
            msg.usage.input_tokens, msg.usage.output_tokens,
            (time.perf_counter()-t0)*1000)

class OpenAIClient(BaseLLMClient):
    def __init__(self, model="gpt-4o-mini", api_key=None):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.chat.completions.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role":"system","content":system},{"role":"user","content":user}])
        c = msg.choices[0]
        return LLMResponse(c.message.content, self.model,
            msg.usage.prompt_tokens, msg.usage.completion_tokens,
            (time.perf_counter()-t0)*1000)

class HuggingFaceClient(BaseLLMClient):
    def __init__(self, model="microsoft/Phi-3-mini-4k-instruct", device="auto"):
        from transformers import pipeline
        self.model_name = model
        self.pipe = pipeline("text-generation", model=model,
                              device_map=device, torch_dtype="auto")
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        msgs = [{"role":"system","content":system},{"role":"user","content":user}]
        t0   = time.perf_counter()
        out  = self.pipe(msgs, max_new_tokens=max_tokens,
                         temperature=max(temperature,1e-4),
                         do_sample=temperature>0, return_full_text=False)
        text = out[0]["generated_text"]
        if isinstance(text, list): text = text[-1]["content"]
        return LLMResponse(text, self.model_name, 0,
                           len(text.split()), (time.perf_counter()-t0)*1000)

def get_llm(backend="claude", **kwargs):
    return {"claude":ClaudeClient,"anthropic":ClaudeClient,
            "openai":OpenAIClient,"gpt":OpenAIClient,
            "hf":HuggingFaceClient,"local":HuggingFaceClient}[backend.lower()](**kwargs)

# ── Deterministic mock with injected noise ────────────────────────────
class MockLLMClient(BaseLLMClient):
    """Mock with configurable accuracy for evaluation experiments."""
    def __init__(self, accuracy: float = 0.80, seed: int = 42):
        self.accuracy = accuracy
        self.rng      = random.Random(seed)
        self._labels  = ["positive", "negative", "neutral", "mixed"]

    def complete(self, system, user, max_tokens=512, temperature=0.0):
        time.sleep(0.02)
        # Extract expected label if embedded in user message
        for label in self._labels:
            if f"label:{label}" in user.lower():
                if self.rng.random() < self.accuracy:
                    text = label.capitalize()
                else:
                    text = self.rng.choice([l for l in self._labels if l != label]).capitalize()
                return LLMResponse(text, "mock-llm",
                                   len((system+user).split()), 1, 25.0)
        # Default
        text = self.rng.choice(self._labels).capitalize()
        return LLMResponse(text, "mock-llm",
                           len((system+user).split()), 1, 25.0)

llm_a = MockLLMClient(accuracy=0.72, seed=1)   # Prompt A (weaker)
llm_b = MockLLMClient(accuracy=0.84, seed=2)   # Prompt B (stronger)
print("Mock LLM clients ready (llm_a=72% acc, llm_b=84% acc).")


## 3. Background: The Evaluation Mindset

### 3.1 The train/dev/test split for prompts

Just as in ML, prompt evaluation requires a principled data split:

```
Full labelled dataset
├── Dev set   (20%)  ← use during prompt development and iteration
├── Test set  (20%)  ← held out; run ONCE to report final numbers
└── Pool      (60%)  ← source of few-shot examples
```

**Critical rule:** never iterate on the test set. Once you look at test-set performance, that information leaks into your prompt design decisions, inflating reported metrics.

---

### 3.2 What makes a good eval dataset

| Property | Description |
|---|---|
| **Representative** | Covers the real distribution of inputs in production |
| **Balanced** | All output classes / difficulty levels are present |
| **Edge-case rich** | Includes adversarial and boundary inputs |
| **Annotator-agreed** | Labels have high inter-annotator agreement |
| **Versioned** | Dataset is frozen and tracked alongside the prompt |

---

### 3.3 Metric taxonomy

$$
\text{Metrics} = \begin{cases}
\text{Exact} & \text{acc, F1, precision, recall} \\
\text{N-gram overlap} & \text{BLEU, ROUGE-N, ROUGE-L} \\
\text{Semantic} & \text{BERTScore, embedding cosine} \\
\text{Model-based} & \text{LLM-as-a-judge, G-Eval}
\end{cases}
$$

Use the **cheapest metric that correlates with human judgement** for your task.


## 4. Building and Managing an Eval Dataset

In [ ]:
@dataclass
class EvalExample:
    """A single labelled evaluation example."""
    id:          str
    input_text:  str
    label:       str                      # ground truth
    metadata:    dict = field(default_factory=dict)
    split:       str  = "dev"             # dev | test | pool

    def __post_init__(self):
        if not self.id:
            # Auto-generate deterministic ID from content
            self.id = hashlib.md5(self.input_text.encode()).hexdigest()[:8]


@dataclass
class EvalDataset:
    """A versioned, split-aware eval dataset."""
    name:     str
    version:  str
    examples: list[EvalExample] = field(default_factory=list)

    def add(self, example: EvalExample):
        self.examples.append(example)

    def split(self, name: str) -> list[EvalExample]:
        return [e for e in self.examples if e.split == name]

    def label_distribution(self, split: str = None) -> dict:
        exs = self.split(split) if split else self.examples
        return dict(Counter(e.label for e in exs))

    def summary(self):
        print(f"Dataset: {self.name} v{self.version}")
        print(f"  Total   : {len(self.examples)}")
        for sp in ["pool", "dev", "test"]:
            exs = self.split(sp)
            if exs:
                dist = Counter(e.label for e in exs)
                print(f"  {sp:<6}  : {len(exs)}  {dict(dist)}")

    def to_jsonl(self, path: str):
        with open(path, "w") as f:
            for ex in self.examples:
                f.write(json.dumps({
                    "id": ex.id, "input": ex.input_text,
                    "label": ex.label, "split": ex.split,
                    "metadata": ex.metadata
                }) + "\n")

    @classmethod
    def from_jsonl(cls, path: str, name: str = "", version: str = "1.0") -> "EvalDataset":
        ds = cls(name=name, version=version)
        with open(path) as f:
            for line in f:
                item = json.loads(line)
                ds.add(EvalExample(
                    id=item["id"], input_text=item["input"],
                    label=item["label"], split=item.get("split","dev"),
                    metadata=item.get("metadata",{})
                ))
        return ds


# ── Build sentiment eval dataset ─────────────────────────────────────
RAW_DATA = [
    # pool examples (for few-shot)
    ("Absolutely love this product, works perfectly!",          "positive", "pool"),
    ("Terrible quality, broke after one day.",                   "negative", "pool"),
    ("Okay for the price, nothing special.",                     "neutral",  "pool"),
    ("Fast delivery but the item was scratched.",                "mixed",    "pool"),
    ("Best purchase I've made this year!",                      "positive", "pool"),
    ("Complete waste of money, avoid.",                          "negative", "pool"),
    ("Does the job, no complaints.",                             "neutral",  "pool"),
    ("Love the design, hate that the battery dies so fast.",     "mixed",    "pool"),
    # dev examples
    ("Five stars — absolutely flawless.",                        "positive", "dev"),
    ("Returned immediately, completely unusable.",               "negative", "dev"),
    ("Neither great nor terrible — mediocre at best.",           "neutral",  "dev"),
    ("Beautiful product but way too expensive.",                 "mixed",    "dev"),
    ("Exceeded all my expectations, stunning quality!",          "positive", "dev"),
    ("Arrived broken and customer service was unhelpful.",       "negative", "dev"),
    ("It works as described, nothing more.",                     "neutral",  "dev"),
    ("Great features, disappointing battery life.",              "mixed",    "dev"),
    # test examples (held-out)
    ("Outstanding quality — will definitely buy again.",         "positive", "test"),
    ("Cheaply made and fell apart within a week.",               "negative", "test"),
    ("Average product, does what it says on the box.",           "neutral",  "test"),
    ("Impressive specs but the build quality lets it down.",     "mixed",    "test"),
    ("Phenomenal value for money, highly recommend!",            "positive", "test"),
    ("Completely misrepresented in the listing.",                "negative", "test"),
    ("Functional but uninspiring.",                              "neutral",  "test"),
    ("Looks premium but feels cheap.",                           "mixed",    "test"),
]

dataset = EvalDataset(name="sentiment-v1", version="1.0")
for text, label, split in RAW_DATA:
    dataset.add(EvalExample(id="", input_text=text, label=label, split=split))

dataset.summary()
print()

# Save & reload (version control pattern)
dataset.to_jsonl("/tmp/sentiment_eval_v1.jsonl")
dataset_loaded = EvalDataset.from_jsonl("/tmp/sentiment_eval_v1.jsonl", "sentiment-v1")
print(f"✅ Saved and reloaded: {len(dataset_loaded.examples)} examples")


## 5. Reference-Based Metrics

### 5.1 Classification metrics

For tasks with a fixed label set (sentiment, intent, etc.):

$$
\text{Accuracy} = \frac{\text{TP} + \text{TN}}{N}
$$

$$
\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c}, \quad
\text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}
$$

$$
\text{F1}_c = \frac{2 \cdot \text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}
$$

**Macro-F1** (unweighted average across classes) is the standard for imbalanced datasets.


In [ ]:
def compute_classification_metrics(
    predictions: list[str],
    references:  list[str],
    labels:      list[str] = None,
) -> dict:
    """
    Compute accuracy, per-class precision/recall/F1, and macro-F1.
    All inputs should be normalised (lowercase, stripped).
    """
    if labels is None:
        labels = sorted(set(references))

    n       = len(predictions)
    correct = sum(p == r for p, r in zip(predictions, references))
    accuracy = correct / n

    per_class = {}
    for label in labels:
        tp = sum(p == label and r == label for p, r in zip(predictions, references))
        fp = sum(p == label and r != label for p, r in zip(predictions, references))
        fn = sum(p != label and r == label for p, r in zip(predictions, references))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        per_class[label] = {"precision": precision, "recall": recall, "f1": f1,
                             "support": sum(r == label for r in references)}

    macro_f1 = np.mean([v["f1"] for v in per_class.values()])
    return {"accuracy": accuracy, "macro_f1": macro_f1, "per_class": per_class, "n": n}


def format_metrics(metrics: dict) -> str:
    lines = [
        f"  Accuracy  : {metrics['accuracy']:.3f}  ({int(metrics['accuracy']*metrics['n'])}/{metrics['n']})",
        f"  Macro-F1  : {metrics['macro_f1']:.3f}",
        f"  {'Label':<12} {'Prec':>6} {'Rec':>6} {'F1':>6} {'N':>5}",
        f"  {'-'*36}",
    ]
    for label, v in metrics["per_class"].items():
        lines.append(f"  {label:<12} {v['precision']:>6.3f} {v['recall']:>6.3f} "
                     f"{v['f1']:>6.3f} {v['support']:>5}")
    return "\n".join(lines)


# Demo
preds = ["positive", "negative", "neutral", "mixed",
         "positive", "negative", "positive", "mixed"]
refs  = ["positive", "negative", "neutral", "positive",
         "positive", "neutral",  "negative", "mixed"]

metrics = compute_classification_metrics(preds, refs)
print("Classification metrics:")
print(format_metrics(metrics))


### 5.2 BLEU and ROUGE

For generation tasks (summarisation, translation, QA):

**BLEU** (Bilingual Evaluation Understudy) measures n-gram precision with a brevity penalty:

$$
\text{BLEU} = \text{BP} \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right)
$$

where $p_n$ is the modified n-gram precision and $\text{BP} = \min(1, e^{1 - r/c})$ penalises short outputs.

**ROUGE-N** measures n-gram recall:

$$
\text{ROUGE-N} = \frac{\sum_{\text{gram}_n \in \text{ref}} \text{count}_{\text{match}}(\text{gram}_n)}{\sum_{\text{gram}_n \in \text{ref}} \text{count}(\text{gram}_n)}
$$

**ROUGE-L** uses longest common subsequence — more robust to word reordering.


In [ ]:
# ── BLEU and ROUGE from scratch ───────────────────────────────────────
from collections import Counter

def ngrams(tokens: list[str], n: int) -> Counter:
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def bleu_score(hypothesis: str, reference: str, max_n: int = 4) -> float:
    """Compute sentence-level BLEU-4 score."""
    hyp_tokens = hypothesis.lower().split()
    ref_tokens = reference.lower().split()

    if len(hyp_tokens) == 0:
        return 0.0

    # Brevity penalty
    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        hyp_ngrams = ngrams(hyp_tokens, n)
        ref_ngrams = ngrams(ref_tokens, n)
        if not hyp_ngrams:
            precisions.append(0.0)
            continue
        matches = sum(min(hyp_ngrams[g], ref_ngrams[g]) for g in hyp_ngrams)
        precisions.append(matches / sum(hyp_ngrams.values()))

    if any(p == 0 for p in precisions):
        return 0.0
    log_avg = sum(math.log(p) / max_n for p in precisions)
    return bp * math.exp(log_avg)


def rouge_n(hypothesis: str, reference: str, n: int = 2) -> dict:
    """Compute ROUGE-N precision, recall, and F1."""
    hyp_tokens = hypothesis.lower().split()
    ref_tokens = reference.lower().split()

    hyp_ng = ngrams(hyp_tokens, n)
    ref_ng = ngrams(ref_tokens, n)

    matches = sum(min(hyp_ng[g], ref_ng[g]) for g in hyp_ng)

    precision = matches / sum(hyp_ng.values()) if hyp_ng else 0.0
    recall    = matches / sum(ref_ng.values()) if ref_ng else 0.0
    f1        = (2*precision*recall/(precision+recall)
                 if (precision+recall) > 0 else 0.0)
    return {"precision": precision, "recall": recall, "f1": f1}


def rouge_l(hypothesis: str, reference: str) -> dict:
    """ROUGE-L using longest common subsequence."""
    hyp = hypothesis.lower().split()
    ref = reference.lower().split()
    m, n = len(hyp), len(ref)
    # LCS dynamic programming
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if hyp[i-1]==ref[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    precision = lcs / m if m > 0 else 0.0
    recall    = lcs / n if n > 0 else 0.0
    f1        = (2*precision*recall/(precision+recall)
                 if (precision+recall) > 0 else 0.0)
    return {"precision": precision, "recall": recall, "f1": f1}


# ── Test on summarisation examples ────────────────────────────────────
pairs = [
    ("The cat sat on the mat.", "The cat sat on the mat."),                # perfect
    ("A cat rested on a mat.", "The cat sat on the mat."),                 # similar
    ("Dogs are loyal animals.", "The cat sat on the mat."),                # different
    ("The quick brown fox jumps over the lazy dog near the river bank.",
     "A fast fox leaps over a sleepy dog."),                               # paraphrase
]

print(f"{'Hypothesis':<45} {'BLEU':>6} {'R-2 F1':>8} {'R-L F1':>8}")
print("-" * 72)
for hyp, ref in pairs:
    b   = bleu_score(hyp, ref)
    r2  = rouge_n(hyp, ref, n=2)["f1"]
    rl  = rouge_l(hyp, ref)["f1"]
    print(f"{hyp[:43]:<45} {b:>6.3f} {r2:>8.3f} {rl:>8.3f}")


## 6. The Prompt Evaluation Harness

In [ ]:
@dataclass
class PromptConfig:
    """A versioned prompt configuration."""
    name:        str
    version:     str
    system:      str
    user_template: str      # use {input} as placeholder
    max_tokens:  int  = 32
    temperature: float = 0.0

    def render(self, input_text: str) -> tuple[str, str]:
        return self.system, self.user_template.format(input=input_text)

    def fingerprint(self) -> str:
        """Deterministic hash of the prompt content."""
        content = self.system + self.user_template
        return hashlib.md5(content.encode()).hexdigest()[:10]


@dataclass
class EvalRun:
    """Results of a single eval run."""
    prompt_name:    str
    prompt_version: str
    prompt_fp:      str
    split:          str
    n_examples:     int
    predictions:    list[str]
    references:     list[str]
    latencies:      list[float]
    token_counts:   list[int]
    metrics:        dict = field(default_factory=dict)
    timestamp:      str  = ""

    def __post_init__(self):
        if not self.timestamp:
            self.timestamp = time.strftime("%Y-%m-%dT%H:%M:%S")
        self.metrics = compute_classification_metrics(
            self.predictions, self.references
        )

    @property
    def accuracy(self): return self.metrics["accuracy"]
    @property
    def macro_f1(self):  return self.metrics["macro_f1"]
    @property
    def avg_latency(self): return np.mean(self.latencies)
    @property
    def avg_tokens(self):  return np.mean(self.token_counts)
    @property
    def cost_estimate(self):
        """Approximate USD cost at Claude Sonnet pricing."""
        return self.n_examples * self.avg_tokens * 3.0 / 1_000_000


class PromptEvaluator:
    """
    Runs prompts against eval datasets and tracks results.
    Supports: single eval, comparative A/B, regression tracking.
    """
    def __init__(self, llm: BaseLLMClient):
        self.llm    = llm
        self.history: list[EvalRun] = []

    def _normalise(self, text: str) -> str:
        """Normalise a prediction for comparison."""
        return text.strip().lower().rstrip(".,!?").split()[0] if text.strip() else ""

    def run(self, prompt: PromptConfig,
            dataset: EvalDataset, split: str = "dev",
            verbose: bool = False) -> EvalRun:
        """Run a prompt against a dataset split and return an EvalRun."""
        examples = dataset.split(split)
        if not examples:
            raise ValueError(f"No examples in split {split!r}")

        predictions, references, latencies, tokens = [], [], [], []

        for ex in examples:
            # Embed expected label for mock client (real clients ignore it)
            user_input = ex.input_text + f" label:{ex.label}"
            system, user = prompt.render(user_input)
            resp = self.llm(system=system, user=user, max_tokens=prompt.max_tokens,
                             temperature=prompt.temperature)
            pred = self._normalise(resp.text)
            predictions.append(pred)
            references.append(ex.label)
            latencies.append(resp.latency_ms)
            tokens.append(resp.total_tokens)

            if verbose:
                match = "✅" if pred == ex.label else "❌"
                print(f"  {match} [{ex.label}→{pred}] {ex.input_text[:45]}")

        run = EvalRun(
            prompt_name=prompt.name, prompt_version=prompt.version,
            prompt_fp=prompt.fingerprint(), split=split,
            n_examples=len(examples),
            predictions=predictions, references=references,
            latencies=latencies, token_counts=tokens,
        )
        self.history.append(run)
        return run

    def summary(self, run: EvalRun):
        print(f"Eval: {run.prompt_name} v{run.prompt_version} [{run.split}]")
        print(f"  Timestamp   : {run.timestamp}")
        print(f"  Fingerprint : {run.prompt_fp}")
        print(format_metrics(run.metrics))
        print(f"  Avg latency : {run.avg_latency:.0f}ms")
        print(f"  Avg tokens  : {run.avg_tokens:.0f}")
        print(f"  Est. cost   : ${run.cost_estimate:.5f} per {run.n_examples} calls")


# ── Define two prompt versions to compare ─────────────────────────────
prompt_v1 = PromptConfig(
    name="sentiment-classifier", version="1.0",
    system="You are a sentiment classifier.",
    user_template="Classify: {input}\nOne word: positive, negative, neutral, or mixed.",
)

prompt_v2 = PromptConfig(
    name="sentiment-classifier", version="2.0",
    system=(
        "You are a precise sentiment analysis assistant. "
        "Classify customer reviews into exactly one of four categories: "
        "positive, negative, neutral, or mixed."
    ),
    user_template=(
        "Review: {input}\n\n"
        "Respond with exactly one word from: positive, negative, neutral, mixed.\n"
        "Word:"
    ),
)

evaluator_a = PromptEvaluator(llm_a)
evaluator_b = PromptEvaluator(llm_b)

run_v1 = evaluator_a.run(prompt_v1, dataset, split="dev")
run_v2 = evaluator_b.run(prompt_v2, dataset, split="dev")

print("=" * 50)
evaluator_a.summary(run_v1)
print()
evaluator_b.summary(run_v2)


## 7. LLM-as-a-Judge

For open-ended tasks where there is no single correct answer (summarisation, explanation quality, tone), reference metrics like BLEU are insufficient.

**LLM-as-a-judge** uses a separate (often larger) model to score outputs on defined criteria:

```
Judge model receives:
  - The original question / input
  - The model's response
  - Scoring rubric (e.g. 1-5 scale with criteria)

Judge outputs:
  - A numeric score
  - A brief justification
```

Key considerations:
- Use a **stronger** model as judge than the one being evaluated
- Provide a **detailed rubric** — vague criteria lead to noisy scores
- Watch for **position bias**: judges often prefer the first response in pairwise comparisons
- Watch for **verbosity bias**: judges often prefer longer responses


In [ ]:
# ── LLM-as-a-Judge implementation ─────────────────────────────────────
JUDGE_RUBRIC = """You are an expert evaluator assessing the quality of AI-generated text.

Score the response on each criterion from 1-5:
  5 = Excellent  4 = Good  3 = Acceptable  2 = Poor  1 = Very poor

Criteria:
  - accuracy:   Is the information factually correct?
  - relevance:  Does it directly answer the question asked?
  - clarity:    Is it clear and easy to understand?
  - conciseness: Is it appropriately concise without losing meaning?

Respond with a JSON object ONLY:
{"accuracy": N, "relevance": N, "clarity": N, "conciseness": N, "justification": "..."}
No markdown fences, no extra text."""


@dataclass
class JudgeScore:
    accuracy:    float
    relevance:   float
    clarity:     float
    conciseness: float
    justification: str
    overall:     float = 0.0

    def __post_init__(self):
        self.overall = np.mean([self.accuracy, self.relevance,
                                self.clarity, self.conciseness])


def llm_judge(
    judge_llm:  BaseLLMClient,
    question:   str,
    response:   str,
    rubric:     str = JUDGE_RUBRIC,
) -> Optional[JudgeScore]:
    """Score a single response using an LLM judge."""
    user = f"Question: {question}\n\nResponse to evaluate:\n{response}"
    resp = judge_llm(system=rubric, user=user, max_tokens=200, temperature=0.0)

    # Parse JSON score
    try:
        raw   = re.sub(r"```(?:json)?\s*", "", resp.text).strip().rstrip("`")
        data  = json.loads(raw)
        return JudgeScore(
            accuracy    = float(data.get("accuracy",    3)),
            relevance   = float(data.get("relevance",   3)),
            clarity     = float(data.get("clarity",     3)),
            conciseness = float(data.get("conciseness", 3)),
            justification = data.get("justification", ""),
        )
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


# ── Mock judge that returns deterministic scores ──────────────────────
class MockJudgeLLM(BaseLLMClient):
    """Returns realistic varying scores for demo."""
    def __init__(self, seed=99):
        self.rng = random.Random(seed)
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        scores = {
            "accuracy":    round(self.rng.uniform(3.0, 5.0), 1),
            "relevance":   round(self.rng.uniform(3.5, 5.0), 1),
            "clarity":     round(self.rng.uniform(2.5, 5.0), 1),
            "conciseness": round(self.rng.uniform(2.0, 4.5), 1),
            "justification": "The response addresses the question adequately."
        }
        text = json.dumps(scores)
        return LLMResponse(text, "mock-judge", 100, 30, 60.0)

judge_llm = MockJudgeLLM(seed=42)

# ── Evaluate several QA responses ────────────────────────────────────
qa_pairs = [
    {
        "question": "What is the difference between supervised and unsupervised learning?",
        "responses": {
            "Verbose":  (
                "Supervised learning is a type of machine learning where the algorithm "
                "learns from labelled training data, meaning each training example has "
                "an associated output label. The model learns to map inputs to outputs. "
                "Unsupervised learning, on the other hand, deals with unlabelled data "
                "and the algorithm must find patterns and structure on its own."
            ),
            "Concise":  (
                "Supervised learning uses labelled data to train models to predict "
                "outputs. Unsupervised learning finds patterns in unlabelled data."
            ),
            "Vague":    (
                "They are both types of machine learning but they work differently "
                "with different kinds of data."
            ),
        }
    }
]

print("LLM-as-a-Judge evaluation:\n")
for qa in qa_pairs:
    q = qa["question"]
    print(f"Question: {q}\n")
    print(f"  {'Response':<12} {'Acc':>5} {'Rel':>5} {'Cla':>5} {'Con':>5} {'Avg':>5}  Justification")
    print(f"  {'-'*75}")
    for name, response in qa["responses"].items():
        score = llm_judge(judge_llm, q, response)
        if score:
            print(f"  {name:<12} {score.accuracy:>5.1f} {score.relevance:>5.1f} "
                  f"{score.clarity:>5.1f} {score.conciseness:>5.1f} "
                  f"{score.overall:>5.2f}  {score.justification[:40]}")


In [ ]:
# ── Pairwise LLM-as-judge (A vs B comparison) ────────────────────────
PAIRWISE_RUBRIC = """You are an expert evaluator comparing two AI responses.

Given a question and two responses (A and B), determine which is better overall.
Consider: accuracy, relevance, clarity, and conciseness.

Respond with JSON only:
{"winner": "A" or "B" or "tie", "confidence": 1-5, "reason": "brief reason"}
No markdown, no extra text."""


def pairwise_judge(judge_llm: BaseLLMClient,
                   question: str, response_a: str, response_b: str,
                   swap: bool = False) -> dict:
    """
    Pairwise A/B judge. Set swap=True to mitigate position bias
    (swap A and B, then invert the result).
    """
    if swap:
        a_label, b_label = "B", "A"
        ra, rb = response_b, response_a
    else:
        a_label, b_label = "A", "B"
        ra, rb = response_a, response_b

    user = (f"Question: {question}\n\n"
            f"Response A:\n{ra}\n\n"
            f"Response B:\n{rb}")
    resp = judge_llm(system=PAIRWISE_RUBRIC, user=user,
                     max_tokens=100, temperature=0.0)
    try:
        raw  = re.sub(r"```(?:json)?\s*", "", resp.text).strip().rstrip("`")
        data = json.loads(raw)
        winner = data.get("winner", "tie")
        # If swapped, invert winner
        if swap:
            winner = {"A": "B", "B": "A", "tie": "tie"}.get(winner, "tie")
        return {"winner": winner,
                "confidence": data.get("confidence", 3),
                "reason": data.get("reason", "")}
    except (json.JSONDecodeError, KeyError):
        return {"winner": "tie", "confidence": 1, "reason": "parse error"}


class MockPairwiseLLM(BaseLLMClient):
    """Mock that slightly prefers response B."""
    def __init__(self, seed=77):
        self.rng = random.Random(seed)
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        # B wins 60% of the time (simulating a better prompt)
        winner = "B" if self.rng.random() < 0.60 else ("A" if self.rng.random() < 0.5 else "tie")
        data = {"winner": winner, "confidence": self.rng.randint(3,5),
                "reason": f"Response {winner} is more clear and accurate."}
        return LLMResponse(json.dumps(data), "mock-pairwise", 150, 20, 55.0)

pairwise_judge_llm = MockPairwiseLLM(seed=42)

# Run 10 pairwise comparisons with position-bias mitigation
question  = "Explain gradient descent in one paragraph."
response_a = "Gradient descent adjusts model weights step by step to minimise the loss."
response_b = "Gradient descent is an optimisation algorithm that iteratively adjusts parameters in the direction of the negative gradient to minimise a loss function."

wins = Counter()
for i in range(10):
    # Alternate swap to mitigate position bias
    result = pairwise_judge(pairwise_judge_llm, question, response_a, response_b,
                             swap=(i % 2 == 1))
    wins[result["winner"]] += 1

print("Pairwise judge results (10 comparisons, position-bias mitigated):")
print(f"  A wins: {wins['A']}  |  B wins: {wins['B']}  |  Ties: {wins['tie']}")
total_decisive = wins["A"] + wins["B"]
if total_decisive > 0:
    pref_b = wins["B"] / (wins["A"] + wins["B"])
    print(f"  Preference for B: {pref_b:.0%}")


## 8. A/B Testing with Statistical Significance

When comparing two prompt versions, you need to know if the observed difference is **real** or just noise.

### McNemar's test

For paired binary outcomes (correct/incorrect on the same examples), **McNemar's test** is the right choice:

$$
\chi^2 = \frac{(b - c)^2}{b + c}
$$

where $b$ = examples correct by B but wrong by A, $c$ = correct by A but wrong by B.

Under $H_0$ (no difference), $\chi^2 \sim \chi^2_1$.

### Bootstrap confidence intervals

A non-parametric alternative: resample with replacement $N$ times, compute the metric each time, and report the 95% interval.


In [ ]:
def mcnemar_test(preds_a: list[str], preds_b: list[str],
                  references: list[str]) -> dict:
    """
    McNemar's test for paired binary outcomes.
    Tests H0: P(A correct) == P(B correct).
    Returns: chi2 statistic, p-value, and interpretation.
    """
    correct_a = [p == r for p, r in zip(preds_a, references)]
    correct_b = [p == r for p, r in zip(preds_b, references)]

    # b: B correct, A wrong  |  c: A correct, B wrong
    b = sum(1 for ca, cb in zip(correct_a, correct_b) if not ca and cb)
    c = sum(1 for ca, cb in zip(correct_a, correct_b) if ca and not cb)
    n = b + c

    if n == 0:
        return {"chi2": 0.0, "p_value": 1.0, "b": b, "c": c,
                "significant": False, "interpretation": "No discordant pairs"}

    # With continuity correction (Yates)
    chi2    = (abs(b - c) - 1)**2 / n if n > 0 else 0.0
    p_value = 1 - stats.chi2.cdf(chi2, df=1)

    return {
        "chi2": chi2, "p_value": p_value, "b": b, "c": c,
        "significant": p_value < 0.05,
        "interpretation": (
            f"B is significantly {'better' if b > c else 'worse'} than A "
            f"(p={p_value:.4f})" if p_value < 0.05
            else f"No significant difference (p={p_value:.4f})"
        )
    }


def bootstrap_ci(values: list[float], n_boot: int = 1000,
                  ci: float = 0.95) -> tuple[float, float]:
    """Non-parametric bootstrap confidence interval."""
    boots = [np.mean(np.random.choice(values, size=len(values), replace=True))
             for _ in np.random.seed(42) or range(n_boot)]
    alpha = (1 - ci) / 2
    return float(np.percentile(boots, alpha*100)), float(np.percentile(boots, (1-alpha)*100))


# ── Run A/B test on dev set ───────────────────────────────────────────
dev_examples = dataset.split("dev")
n = len(dev_examples)
refs = [e.label for e in dev_examples]

# Simulate predictions from two prompt versions
rng_a = random.Random(10)
rng_b = random.Random(20)
labels = ["positive", "negative", "neutral", "mixed"]

preds_a = [e.label if rng_a.random() < 0.72 else rng_a.choice(labels) for e in dev_examples]
preds_b = [e.label if rng_b.random() < 0.84 else rng_b.choice(labels) for e in dev_examples]

acc_a = sum(p==r for p,r in zip(preds_a,refs)) / n
acc_b = sum(p==r for p,r in zip(preds_b,refs)) / n

result = mcnemar_test(preds_a, preds_b, refs)

print("A/B Test Results")
print(f"  Prompt A accuracy : {acc_a:.3f}  ({int(acc_a*n)}/{n})")
print(f"  Prompt B accuracy : {acc_b:.3f}  ({int(acc_b*n)}/{n})")
print(f"  Δ accuracy        : {acc_b-acc_a:+.3f}  ({(acc_b-acc_a)*100:+.1f}pp)")
print()
print("McNemar's test:")
print(f"  χ² = {result['chi2']:.3f}  |  p = {result['p_value']:.4f}")
print(f"  b (B right, A wrong) = {result['b']}")
print(f"  c (A right, B wrong) = {result['c']}")
print(f"  Result: {result['interpretation']}")


In [ ]:
# ── Bootstrap CI plot ────────────────────────────────────────────────
np.random.seed(42)
n_boot = 1000

correct_a = [float(p==r) for p,r in zip(preds_a, refs)]
correct_b = [float(p==r) for p,r in zip(preds_b, refs)]

boots_a = [np.mean(np.random.choice(correct_a, size=n, replace=True)) for _ in range(n_boot)]
boots_b = [np.mean(np.random.choice(correct_b, size=n, replace=True)) for _ in range(n_boot)]

ci_a = (np.percentile(boots_a, 2.5), np.percentile(boots_a, 97.5))
ci_b = (np.percentile(boots_b, 2.5), np.percentile(boots_b, 97.5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("A/B Test: Bootstrap Confidence Intervals", fontsize=12)

# Distribution of bootstrap accuracy
for ax_idx, (boots, ci, label, color) in enumerate([
    (boots_a, ci_a, "Prompt A", "#3498db"),
    (boots_b, ci_b, "Prompt B", "#2ecc71"),
]):
    axes[0].hist(boots, bins=30, alpha=0.6, color=color, label=f"{label} ({np.mean(boots):.3f})")
    axes[0].axvline(ci[0], color=color, linestyle=":", lw=1.5)
    axes[0].axvline(ci[1], color=color, linestyle=":", lw=1.5)

axes[0].set_xlabel("Accuracy")
axes[0].set_ylabel("Bootstrap frequency")
axes[0].set_title("Bootstrap distribution of accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Delta distribution
delta_boots = [b - a for a, b in zip(boots_a, boots_b)]
ci_delta    = (np.percentile(delta_boots, 2.5), np.percentile(delta_boots, 97.5))
color_delta = "#2ecc71" if ci_delta[0] > 0 else ("#e74c3c" if ci_delta[1] < 0 else "#f39c12")
axes[1].hist(delta_boots, bins=30, color=color_delta, alpha=0.7)
axes[1].axvline(0, color="black", lw=1.5, linestyle="--", label="No difference")
axes[1].axvline(ci_delta[0], color="gray", linestyle=":", lw=1.5)
axes[1].axvline(ci_delta[1], color="gray", linestyle=":", lw=1.5)
axes[1].set_xlabel("Accuracy delta (B - A)")
axes[1].set_ylabel("Bootstrap frequency")
axes[1].set_title(f"Δ accuracy 95% CI: [{ci_delta[0]:+.3f}, {ci_delta[1]:+.3f}]")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

overlap = ci_delta[0] > 0
print(f"95% CI for Δ accuracy: [{ci_delta[0]:+.3f}, {ci_delta[1]:+.3f}]")
print(f"CI excludes zero: {'✅ B is significantly better' if overlap else '⚠️  Not conclusive'}")


## 9. Regression Testing and Prompt Versioning

In [ ]:
# ── Regression test suite ─────────────────────────────────────────────
@dataclass
class RegressionSuite:
    """
    Tracks prompt performance across versions.
    Alerts when accuracy drops below a threshold.
    """
    name:      str
    threshold: float = 0.75    # minimum acceptable accuracy
    runs:      list[EvalRun] = field(default_factory=list)

    def add_run(self, run: EvalRun):
        self.runs.append(run)

    def check_regression(self, run: EvalRun) -> dict:
        """Check if this run regresses vs the best previous run."""
        if len(self.runs) < 2:
            return {"regression": False, "message": "Not enough history"}

        prev_runs = [r for r in self.runs[:-1] if r.split == run.split]
        if not prev_runs:
            return {"regression": False, "message": "No previous runs for this split"}

        best_prev = max(prev_runs, key=lambda r: r.accuracy)
        delta     = run.accuracy - best_prev.accuracy

        regression = run.accuracy < self.threshold or delta < -0.05  # 5pp drop

        return {
            "regression":   regression,
            "current_acc":  run.accuracy,
            "best_prev_acc": best_prev.accuracy,
            "delta":        delta,
            "message": (
                f"⚠️  REGRESSION: {delta:+.3f} ({run.accuracy:.3f} < {best_prev.accuracy:.3f})"
                if regression else
                f"✅ OK: {delta:+.3f} ({run.accuracy:.3f} vs {best_prev.accuracy:.3f})"
            )
        }

    def plot_history(self):
        if not self.runs:
            return
        versions = [f"v{r.prompt_version}" for r in self.runs]
        accs     = [r.accuracy for r in self.runs]
        colors   = ["#2ecc71" if a >= self.threshold else "#e74c3c" for a in accs]

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(range(len(versions)), accs, color=colors, edgecolor="white")
        ax.axhline(self.threshold, color="#e74c3c", linestyle="--",
                   lw=1.5, label=f"Min threshold ({self.threshold:.0%})")
        ax.set_xticks(range(len(versions)))
        ax.set_xticklabels(versions, rotation=30)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Accuracy")
        ax.set_title(f"Regression suite: {self.name}")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        for i, (v, a) in enumerate(zip(versions, accs)):
            ax.text(i, a + 0.01, f"{a:.2f}", ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()


# ── Simulate version history ──────────────────────────────────────────
suite = RegressionSuite(name="sentiment-classifier", threshold=0.75)

version_accs = [0.70, 0.78, 0.82, 0.84, 0.76, 0.88]  # v6 has a small dip
for i, acc in enumerate(version_accs, 1):
    rng = random.Random(i * 13)
    n   = len(dataset.split("dev"))
    preds_sim = [e.label if rng.random() < acc else rng.choice(labels)
                 for e in dataset.split("dev")]
    run_sim = EvalRun(
        prompt_name="sentiment-classifier", prompt_version=f"{i}.0",
        prompt_fp=f"fp{i:04d}", split="dev", n_examples=n,
        predictions=preds_sim, references=[e.label for e in dataset.split("dev")],
        latencies=[25.0]*n, token_counts=[35]*n,
    )
    suite.add_run(run_sim)

# Check latest run for regression
check = suite.check_regression(suite.runs[-1])
print("Regression check (latest run):")
print(f"  {check['message']}")
print()

suite.plot_history()


## 10. The Prompt Evaluation Pyramid

A practical framework for scaling evaluation effort:

```
                    ▲
                   ███          Human evaluation
                  █████         (gold standard, expensive)
                 ███████
                ─────────
               █████████        LLM-as-a-judge
              ███████████       (flexible, moderate cost)
             █████████████
            ───────────────
           ███████████████████  Reference metrics: BLEU, ROUGE
          █████████████████████ (fast, cheap, limited to generation)
         ─────────────────────────────────
        █████████████████████████████████  Exact match / classification metrics
       ███████████████████████████████████ (fastest, cheapest, requires labels)
      ─────────────────────────────────────────────────────────────────────
```

**Engineering principle:** start at the bottom. Build fast exact-match evals first.
Add LLM-judge only for tasks where deterministic metrics are insufficient.
Reserve human eval for launch decisions and benchmark calibration.


In [ ]:
# ── Cost/speed/coverage tradeoff ─────────────────────────────────────
eval_methods = {
    "Exact match":        {"speed_ms": 0.1,  "cost_per_1k": 0.00, "task_coverage": 0.40},
    "F1 / classification":{"speed_ms": 0.5,  "cost_per_1k": 0.00, "task_coverage": 0.55},
    "BLEU / ROUGE":       {"speed_ms": 2.0,  "cost_per_1k": 0.00, "task_coverage": 0.65},
    "BERTScore":          {"speed_ms": 50.0, "cost_per_1k": 0.00, "task_coverage": 0.75},
    "LLM-judge (small)":  {"speed_ms": 200,  "cost_per_1k": 0.15, "task_coverage": 0.88},
    "LLM-judge (large)":  {"speed_ms": 800,  "cost_per_1k": 3.00, "task_coverage": 0.95},
    "Human eval":         {"speed_ms": 30000,"cost_per_1k": 500,  "task_coverage": 1.00},
}

methods  = list(eval_methods.keys())
speeds   = [eval_methods[m]["speed_ms"]      for m in methods]
costs    = [eval_methods[m]["cost_per_1k"]   for m in methods]
coverage = [eval_methods[m]["task_coverage"] for m in methods]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Eval Method Comparison: Speed · Cost · Coverage", fontsize=12)

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(methods)))

for ax, vals, ylabel, title, log in [
    (axes[0], speeds,   "ms per example (log)",   "Eval speed",    True),
    (axes[1], costs,    "USD per 1000 examples",  "Eval cost",     False),
    (axes[2], coverage, "Fraction of tasks covered","Task coverage", False),
]:
    bars = ax.barh(methods, vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_xlabel(ylabel)
    ax.set_title(title)
    if log and all(v > 0 for v in vals):
        ax.set_xscale("log")
    ax.grid(axis="x", alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(max(vals)*0.02 + bar.get_width(), bar.get_y() + bar.get_height()/2,
                f"{v}", va="center", fontsize=8)

plt.tight_layout()
plt.show()


## 11. Putting It All Together: Production Eval Pipeline

In [ ]:
class PromptEvalPipeline:
    """
    End-to-end prompt evaluation pipeline.
    Combines: reference eval + LLM-judge + A/B testing + regression tracking.
    """
    def __init__(self,
                 task_llm:   BaseLLMClient,
                 judge_llm:  BaseLLMClient,
                 dataset:    EvalDataset,
                 suite:      RegressionSuite):
        self.task_llm  = task_llm
        self.judge_llm = judge_llm
        self.dataset   = dataset
        self.suite     = suite
        self.evaluator = PromptEvaluator(task_llm)

    def evaluate_prompt(self, prompt: PromptConfig,
                         split: str = "dev",
                         run_judge: bool = False,
                         judge_sample: int = 3) -> dict:
        """Full evaluation of a single prompt."""
        # 1. Reference-based metrics
        run = self.evaluator.run(prompt, self.dataset, split=split)

        # 2. Regression check
        self.suite.add_run(run)
        reg = self.suite.check_regression(run)

        # 3. Optional LLM-judge on sample
        judge_scores = []
        if run_judge:
            examples = self.dataset.split(split)[:judge_sample]
            for ex in examples:
                _, user = prompt.render(ex.input_text)
                pred_idx = self.dataset.split(split).index(ex)
                response = run.predictions[pred_idx] if pred_idx < len(run.predictions) else ""
                score = llm_judge(self.judge_llm, ex.input_text, response)
                if score:
                    judge_scores.append(score.overall)

        return {
            "run":          run,
            "regression":   reg,
            "judge_avg":    np.mean(judge_scores) if judge_scores else None,
            "judge_n":      len(judge_scores),
        }

    def compare_prompts(self, prompt_a: PromptConfig,
                         prompt_b: PromptConfig,
                         split: str = "dev") -> dict:
        """A/B comparison with statistical significance test."""
        run_a = self.evaluator.run(prompt_a, self.dataset, split=split)
        run_b = self.evaluator.run(prompt_b, self.dataset, split=split)

        mc = mcnemar_test(run_a.predictions, run_b.predictions, run_a.references)

        return {
            "prompt_a":   {"name": prompt_a.name, "version": prompt_a.version,
                           "accuracy": run_a.accuracy, "macro_f1": run_a.macro_f1},
            "prompt_b":   {"name": prompt_b.name, "version": prompt_b.version,
                           "accuracy": run_b.accuracy, "macro_f1": run_b.macro_f1},
            "mcnemar":    mc,
            "delta_acc":  run_b.accuracy - run_a.accuracy,
            "winner":     ("B" if mc["significant"] and run_b.accuracy > run_a.accuracy
                           else "A" if mc["significant"] else "no_significant_diff"),
        }


# ── Run the full pipeline ─────────────────────────────────────────────
pipeline = PromptEvalPipeline(
    task_llm=llm_b, judge_llm=judge_llm,
    dataset=dataset, suite=suite
)

print("Single prompt evaluation:")
result_v2 = pipeline.evaluate_prompt(prompt_v2, split="dev",
                                      run_judge=True, judge_sample=3)
print(f"  Accuracy : {result_v2['run'].accuracy:.3f}")
print(f"  Macro-F1 : {result_v2['run'].macro_f1:.3f}")
print(f"  Regression: {result_v2['regression']['message']}")
if result_v2["judge_avg"]:
    print(f"  Judge avg: {result_v2['judge_avg']:.2f}/5.0 (n={result_v2['judge_n']})")

print()
print("A/B comparison:")
ab = pipeline.compare_prompts(prompt_v1, prompt_v2, split="dev")
print(f"  A ({ab['prompt_a']['version']}): acc={ab['prompt_a']['accuracy']:.3f}")
print(f"  B ({ab['prompt_b']['version']}): acc={ab['prompt_b']['accuracy']:.3f}")
print(f"  Δ = {ab['delta_acc']:+.3f}  |  {ab['mcnemar']['interpretation']}")
print(f"  Winner: {ab['winner']}")


## 12. Engineering Insights

### 12.1 Eval dataset sizing

How many examples do you need to detect a real improvement?

For a two-sided test at $\alpha = 0.05$, power $= 0.80$:

| Baseline acc | Min detectable $\Delta$ | Required $n$ |
|---|---|---|
| 0.70 | 0.10 | ~60 |
| 0.80 | 0.05 | ~200 |
| 0.90 | 0.03 | ~500 |
| 0.95 | 0.02 | ~1200 |

**Rule of thumb:** 100–200 examples detects a ~5pp improvement reliably.

### 12.2 Eval cost management

| Method | Cost at 200 examples |
|---|---|
| Exact match | $0.00 |
| BLEU/ROUGE | $0.00 |
| LLM-judge (Haiku) | ~$0.05 |
| LLM-judge (Sonnet) | ~$0.60 |
| LLM-judge (Opus) | ~$6.00 |

Use cheap models for continuous CI evals; reserve expensive judges for pre-launch review.

### 12.3 The eval golden rules

1. **Lock the test set** — never iterate against it; it exists for reporting only
2. **Version everything** — dataset JSONL, prompt text, model name, eval date
3. **Automate in CI** — run evals on every prompt or model change
4. **Report confidence intervals** — a single number without uncertainty is meaningless
5. **Monitor production** — eval on static sets ≠ real-world performance; log live outputs
6. **Align with human judgment** — periodically validate that your metric correlates with actual user satisfaction


## 13. Exercises

1. **Dataset construction:** Build a 50-example eval dataset for a task of your choice (e.g. intent classification, toxicity detection, question answering). Ensure balanced class distribution and include at least 10 edge-case examples. Save as JSONL.

2. **Metric correlation:** For 20 summarisation examples, compute BLEU-4, ROUGE-2, and ROUGE-L. Ask a human (or yourself) to rate each summary 1-5. Which metric correlates most strongly with human ratings? Compute Spearman's $\rho$.

3. **LLM-judge rubric design:** Write a scoring rubric for evaluating code generation responses (correctness, readability, efficiency). Run your judge on 5 code snippets. Compare the judge's rankings to your own intuition.

4. **Statistical power:** Using the `mcnemar_test` function, simulate 1000 A/B experiments where the true accuracy difference is exactly 5pp. What fraction of tests correctly detect the difference (power)? How does it change with dataset size (n=50, 100, 200)?

5. **CI integration:** Write a `run_eval_ci.py` script (or notebook function) that: loads a prompt and dataset from files, runs the eval, compares against a threshold in a config file, and exits with code 1 if a regression is detected. Design it to run in a GitHub Actions workflow.

---

## 14. Key Takeaways

| Concept | Summary |
|---|---|
| **Eval mindset** | Measurement over intuition — always have numbers |
| **Data splits** | Pool / dev / test — never iterate against test |
| **Exact match / F1** | Fastest, cheapest — use first for classification |
| **BLEU / ROUGE** | N-gram overlap — adequate for generation tasks |
| **LLM-as-a-judge** | Flexible, scalable — watch for position/verbosity bias |
| **Pairwise judge** | Swap A/B positions to mitigate bias |
| **McNemar's test** | Statistical significance for paired binary outcomes |
| **Bootstrap CI** | Non-parametric confidence intervals — always report them |
| **Regression suite** | Track performance across versions — alert on drops |
| **Eval pyramid** | Match eval cost to task type — start cheap, scale up |

---

🎉 **Module 02: Prompt Engineering — Complete!**

All 5 notebooks:
- `01_zero_shot_prompting.ipynb` ✅
- `02_few_shot_prompting.ipynb` ✅
- `03_chain_of_thought.ipynb` ✅
- `04_structured_prompting.ipynb` ✅
- `05_prompt_evaluation.ipynb` ✅

**Next module →** `03_embeddings_and_retrieval/` — how to build semantic search and retrieval systems.
